In [ ]:
import os
from glob import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Klassen aus deinem Screenshot
# ---------------------------------------------------------
CLASS_NAMES = [
    "double_plant",
    "drydown",
    "endrow",
    "nutrient_deficiency",
    "planter_skip",
    "storm_damage",
    "water",
    "waterway",
    "weed_cluster",
]

NUM_CLASSES = len(CLASS_NAMES)
IN_CHANNELS = 3
DEBUG = True  # kannst du später auf False setzen


# ---------------------------------------------------------
# 1) Dataset
# ---------------------------------------------------------
class AgricultureVisionMultiLabel(Dataset):
    def __init__(self, root, split="train", transform=None, debug=False):
        self.root = root
        self.split = split
        self.transform = transform
        self.debug = debug

        base = os.path.join(root, split)

        self.rgb_dir = os.path.join(base, "images", "rgb")
        self.boundary_dir = os.path.join(base, "boundaries")
        self.mask_dir = os.path.join(base, "masks")

        # Label Ordner pro Klasse
        self.label_dirs = {
            cls: os.path.join(base, "labels", cls) for cls in CLASS_NAMES
        }

        # Masterliste aller RGB Bilder
        self.samples = sorted(glob(os.path.join(self.rgb_dir, "*.*")))
        if len(self.samples) == 0:
            raise RuntimeError(f"Keine RGB Bilder in: {self.rgb_dir}")

        if self.debug:
            print(f"\n[INIT] Split = {split}")
            print(f"RGB Bilder gefunden: {len(self.samples)}")
            print("Klassen:", CLASS_NAMES)

    def __len__(self):
        return len(self.samples)

    def _id_from_path(self, p):
        return os.path.splitext(os.path.basename(p))[0]

    def __getitem__(self, idx):
        rgb_path = self.samples[idx]
        sample_id = self._id_from_path(rgb_path)

        boundary_path = os.path.join(self.boundary_dir, sample_id + ".png")
        mask_path = os.path.join(self.mask_dir, sample_id + ".png")

        # RGB
        rgb = Image.open(rgb_path).convert("RGB")
        rgb = np.array(rgb, dtype=np.float32) / 255.0  # [H,W,3]

        # Boundary
        boundary = Image.open(boundary_path).convert("L")
        boundary = np.array(boundary, dtype=np.uint8)

        # Farm Mask
        farm_mask = Image.open(mask_path).convert("L")
        farm_mask = np.array(farm_mask, dtype=np.uint8)

        # Gültige Pixel: innerhalb boundary und innerhalb farm_mask
        valid_mask = (boundary > 0) & (farm_mask > 0)  # bool [H,W]

        # Label Masken pro Klasse
        labels = []
        for cls in CLASS_NAMES:
            cls_path = os.path.join(self.label_dirs[cls], sample_id + ".png")

            if os.path.exists(cls_path):
                lbl = Image.open(cls_path).convert("L")
                lbl = np.array(lbl, dtype=np.uint8)
            else:
                # falls für diese Klasse keine Datei existiert
                lbl = np.zeros_like(valid_mask, dtype=np.uint8)

            labels.append(lbl > 0)  # binär

        labels = np.stack(labels, axis=0).astype(np.float32)  # [9,H,W]

        # ungültige Pixel komplett auf 0 setzen
        labels[:, ~valid_mask] = 0.0

        # nach Torch konvertieren
        rgb = torch.from_numpy(rgb).permute(2, 0, 1)          # [3,H,W]
        labels = torch.from_numpy(labels).float()             # [9,H,W]
        valid_mask = torch.from_numpy(valid_mask).bool()      # [H,W]
        boundary = torch.from_numpy(boundary)
        farm_mask = torch.from_numpy(farm_mask)

        sample = {
            "image": rgb,
            "labels": labels,
            "valid_mask": valid_mask,
            "id": sample_id,
            "boundary": boundary,
            "mask": farm_mask,
        }

        if self.transform:
            sample = self.transform(sample)

        return sample


# ---------------------------------------------------------
# 2) Identity Transform
# ---------------------------------------------------------
class IdentityTransform:
    def __call__(self, s):
        return s


# ---------------------------------------------------------
# 3) Loader und Test Batch
# ---------------------------------------------------------
train_dataset = AgricultureVisionMultiLabel("data", "train", IdentityTransform(), debug=DEBUG)
train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)

batch = next(iter(train_loader))

img = batch["image"][0].permute(1, 2, 0).numpy()
boundary = batch["boundary"][0].numpy()
farm_mask = batch["mask"][0].numpy()
valid_mask = batch["valid_mask"][0].numpy()
labels = batch["labels"][0].numpy()
sample_id = batch["id"][0]

print("\n[BATCH INFO]")
print("ID:", sample_id)
print("Image shape:", img.shape)
print("Labels shape:", labels.shape)
print("Valid mask shape:", valid_mask.shape)



import matplotlib.pyplot as plt

def visualize_sample(sample, class_names):
    """
    sample = output from DataLoader batch[0]
    class_names = CLASS_NAMES list
    """

    rgb = sample["image"].permute(1,2,0).numpy()
    boundary = sample["boundary"].numpy()
    mask = sample["mask"].numpy()
    valid_mask = sample["valid_mask"].numpy()
    labels = sample["labels"].numpy()
    id = sample["id"]

    plt.figure(figsize=(18, 18))

    # -------------------------------
    # 1) RGB + Boundary + Mask + Valid Mask
    # -------------------------------
    print(id)
    plt.subplot(4, 4, 1)
    plt.title("RGB")
    plt.imshow(rgb)

    plt.subplot(4, 4, 2)
    plt.title("Boundary")
    plt.imshow(boundary, cmap="gray", vmin=0, vmax=255)

    plt.subplot(4, 4, 3)
    plt.title("Mask (Farm)")
    plt.imshow(mask, cmap="gray", vmin=0, vmax=255)

    plt.subplot(4, 4, 4)
    plt.title("Valid (boundary ∧ mask)")
    plt.imshow(valid_mask, cmap="gray")

    # -------------------------------
    # 2) 9 Klassenmasken
    # -------------------------------
    for i, cls in enumerate(class_names):
        plt.subplot(4, 4, 5 + i)
        plt.title(cls)
        plt.imshow(labels[i], cmap="gray")

    # Leere Plots (bei 16 Slots – 13 Bilder)
    for empty in range(5 + len(class_names), 17):
        plt.subplot(4, 4, empty)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

batch = next(iter(train_loader))

sample = {
    "image": batch["image"][0],
    "boundary": batch["boundary"][0],
    "mask": batch["mask"][0],
    "valid_mask": batch["valid_mask"][0],
    "labels": batch["labels"][0],
    "id": batch["id"][0]
}

visualize_sample(sample, CLASS_NAMES)


ModuleNotFoundError: No module named 'numpy'

In [2]:
import os
from glob import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import random
import torchvision.transforms.functional as TF

# ---------------------------------------------------------
# Klassen aus deinem Screenshot
# ---------------------------------------------------------
CLASS_NAMES = [
    "double_plant",
    "drydown",
    "endrow",
    "nutrient_deficiency",
    "planter_skip",
    "storm_damage",
    "water",
    "waterway",
    "weed_cluster",
]

NUM_CLASSES = len(CLASS_NAMES)
IN_CHANNELS = 3
DEBUG = True  # kannst du später auf False setzen

# ---------------------------------------------------------
# Normalisierung für RGB-Kanäle (ImageNet Standard)
# ---------------------------------------------------------
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

# ---------------------------------------------------------
# 1) Dataset
# ---------------------------------------------------------
class AgricultureVisionMultiLabel(Dataset):
    def __init__(self, root, split="train", transform=None, debug=False):
        self.root = root
        self.split = split
        self.transform = transform
        self.debug = debug

        base = os.path.join(root, split)

        self.rgb_dir = os.path.join(base, "images", "rgb")
        self.boundary_dir = os.path.join(base, "boundaries")
        self.mask_dir = os.path.join(base, "masks")

        # Label Ordner pro Klasse
        self.label_dirs = {
            cls: os.path.join(base, "labels", cls) for cls in CLASS_NAMES
        }

        # Masterliste aller RGB Bilder
        self.samples = sorted(glob(os.path.join(self.rgb_dir, "*.*")))
        if len(self.samples) == 0:
            raise RuntimeError(f"Keine RGB Bilder in: {self.rgb_dir}")

        if self.debug:
            print(f"\n[INIT] Split = {split}")
            print(f"RGB Bilder gefunden: {len(self.samples)}")
            print("Klassen:", CLASS_NAMES)

    def __len__(self):
        return len(self.samples)

    def _id_from_path(self, p):
        return os.path.splitext(os.path.basename(p))[0]

    def __getitem__(self, idx):
        rgb_path = self.samples[idx]
        sample_id = self._id_from_path(rgb_path)

        boundary_path = os.path.join(self.boundary_dir, sample_id + ".png")
        mask_path = os.path.join(self.mask_dir, sample_id + ".png")

        # RGB
        rgb = Image.open(rgb_path).convert("RGB")
        rgb = np.array(rgb, dtype=np.float32) / 255.0  # [H,W,3]

        # Boundary
        boundary = Image.open(boundary_path).convert("L")
        boundary = np.array(boundary, dtype=np.uint8)

        # Farm Mask
        farm_mask = Image.open(mask_path).convert("L")
        farm_mask = np.array(farm_mask, dtype=np.uint8)

        # Gültige Pixel: innerhalb boundary und innerhalb farm_mask
        valid_mask = (boundary > 0) & (farm_mask > 0)  # bool [H,W]

        # Label Masken pro Klasse
        labels = []
        for cls in CLASS_NAMES:
            cls_path = os.path.join(self.label_dirs[cls], sample_id + ".png")

            if os.path.exists(cls_path):
                lbl = Image.open(cls_path).convert("L")
                lbl = np.array(lbl, dtype=np.uint8)
            else:
                lbl = np.zeros_like(valid_mask, dtype=np.uint8)

            labels.append(lbl > 0)

        labels = np.stack(labels, axis=0).astype(np.float32)  # [9,H,W]

        # ungültige Pixel auf 0 setzen
        labels[:, ~valid_mask] = 0.0

        # Torch konvertieren
        rgb = torch.from_numpy(rgb).permute(2, 0, 1)          # [3,H,W]
        labels = torch.from_numpy(labels).float()             # [9,H,W]
        valid_mask = torch.from_numpy(valid_mask).bool()      # [H,W]
        boundary = torch.from_numpy(boundary)
        farm_mask = torch.from_numpy(farm_mask)

        sample = {
            "image": rgb,
            "labels": labels,
            "valid_mask": valid_mask,
            "id": sample_id,
            "boundary": boundary,
            "mask": farm_mask,
        }

        # TRANSFORM (PREPROCESSING) HIER
        if self.transform:
            sample = self.transform(sample)

        return sample


# ---------------------------------------------------------
# 2) Identity Transform
# ---------------------------------------------------------
class IdentityTransform:
    def __call__(self, s):
        return s


# ---------------------------------------------------------
# 3) Basic Preprocessing (Schritt B)
# ---------------------------------------------------------
class BasicTransform:
    def __call__(self, sample):
        img = sample["image"].clone()

        # Normalisierung: (x - mean) / std
        img = (img - MEAN) / STD

        sample["image"] = img
        return sample


# ---------------------------------------------------------
# 4) Optional: Augmentations (für später)
# ---------------------------------------------------------
class AugmentTransform:
    def __call__(self, sample):

        img = sample["image"]
        labels = sample["labels"]
        boundary = sample["boundary"]
        mask = sample["mask"]
        valid_mask = sample["valid_mask"]

        # horizontales Flippen
        if random.random() < 0.5:
            img = TF.hflip(img)
            labels = TF.hflip(labels)
            boundary = TF.hflip(boundary)
            mask = TF.hflip(mask)
            valid_mask = TF.hflip(valid_mask)

        sample["image"] = img
        sample["labels"] = labels
        sample["boundary"] = boundary
        sample["mask"] = mask
        sample["valid_mask"] = valid_mask

        return sample


# ---------------------------------------------------------
# 5) Loader und Preprocessing aktivieren
# ---------------------------------------------------------

train_dataset = AgricultureVisionMultiLabel(
    "data", 
    "train",
    transform=BasicTransform(),   # Preprocessing aktiv
    debug=DEBUG
)

val_dataset = AgricultureVisionMultiLabel(
    "data", 
    "val",
    transform=BasicTransform(),
    debug=False
)

test_dataset = AgricultureVisionMultiLabel(
    "data", 
    "test",
    transform=BasicTransform(),
    debug=False
)

train_loader = DataLoader(train_dataset, batch_size=200, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=200, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=200, shuffle=False)


# ---------------------------------------------------------
# 6) Test Batch ausgeben
# ---------------------------------------------------------
batch = next(iter(train_loader))

img = batch["image"][0].permute(1, 2, 0).numpy()
boundary = batch["boundary"][0].numpy()
farm_mask = batch["mask"][0].numpy()
valid_mask = batch["valid_mask"][0].numpy()
labels = batch["labels"][0].numpy()
sample_id = batch["id"][0]

print("\n[BATCH INFO]")
print("ID:", sample_id)
print("Image shape:", img.shape)
print("Labels shape:", labels.shape)
print("Valid mask shape:", valid_mask.shape)



[INIT] Split = train
RGB Bilder gefunden: 56944
Klassen: ['double_plant', 'drydown', 'endrow', 'nutrient_deficiency', 'planter_skip', 'storm_damage', 'water', 'waterway', 'weed_cluster']

[BATCH INFO]
ID: YUTW63DUX_13621-1109-14133-1621
Image shape: (512, 512, 3)
Labels shape: (9, 512, 512)
Valid mask shape: (512, 512)


In [3]:
#model
import torch
import torch.nn as nn
from torchvision.models.segmentation import deeplabv3_resnet50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


class DeepLabMultiLabel(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()

        # torchvision DeepLabV3+ (ResNet50 Backbone)
        self.model = deeplabv3_resnet50(weights=None, weights_backbone=None)

        # finale Klassifikation ersetzen (21 -> 9)
        self.model.classifier[-1] = nn.Conv2d(
            in_channels=256,
            out_channels=num_classes,
            kernel_size=1
        )

    def forward(self, x):
        out = self.model(x)["out"]  # [B,9,H,W]
        return out


model = DeepLabMultiLabel(num_classes=9).to(device)
print("Modell geladen.")

Device: cpu
Modell geladen.


In [4]:
#loss function
bce = nn.BCEWithLogitsLoss(reduction="none")

def multilabel_loss(logits, labels, valid_mask):
    """
    logits: [B,9,H,W]
    labels: [B,9,H,W]
    valid_mask: [B,H,W]
    """

    raw = bce(logits, labels)            # [B,9,H,W]

    mask = valid_mask.unsqueeze(1)       # [B,1,H,W]
    masked = raw * mask                  # ungültige Pixel = 0

    denom = mask.sum() + 1e-6            # Anzahl gültiger Pixel
    loss = masked.sum() / denom

    return loss

In [5]:
#optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

In [6]:
def train_one_epoch(model, loader, optimizer, epoch=0):
    model.train()
    total_loss = 0

    print(f"\n=== Training Epoch {epoch+1} ===")

    for batch_idx, batch in enumerate(loader):

        imgs        = batch["image"].to(device)
        labels      = batch["labels"].to(device)
        valid_mask  = batch["valid_mask"].to(device)
        ids         = batch["id"]        # Liste mit Bild-IDs

        # Training Schritt
        optimizer.zero_grad()
        logits = model(imgs)
        loss = multilabel_loss(logits, labels, valid_mask)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # ----------------------------
        # LOGGING: Fortschritt + Bild-IDs
        # ----------------------------
        print(f"Batch {batch_idx+1}/{len(loader)} | "
              f"Loss: {loss.item():.4f} | "
              f"Images: {', '.join(ids)}")

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} Finished — Avg Loss: {avg_loss:.4f}")
    return avg_loss

In [ ]:
def validate(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in loader:

            imgs = batch["image"].to(device)
            labels = batch["labels"].to(device)
            valid_mask = batch["valid_mask"].to(device)

            logits = model(imgs)
            loss = multilabel_loss(logits, labels, valid_mask)
            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
EPOCHS = 1

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss = validate(model, val_loader)

    scheduler.step()

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


=== Training Epoch 1 ===


In [ ]:
model.eval()
batch = next(iter(val_loader))

imgs = batch["image"].to(device)
valid = batch["valid_mask"]

with torch.no_grad():
    logits = model(imgs)
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

print("Prediction shape:", preds.shape)